# DAIC-WOZ Preprocessing

## Unzip the dataset and keep necessary files

In [2]:
import zipfile
import os
import shutil

In [3]:
# Unzip the downloaded zip files and only keep the .wav and TRANSCRIPT.csv files
import zipfile
import os
import shutil

# Define the directory where the zip files are stored
zip_dir = '../../data/raw/DAIC_WOZ/'

# Define the directory where the unzipped files will be stored
unzip_dir = '../../data/raw/DAIC_WOZ_unzipped/'

# Delete the directory if it already exists
shutil.rmtree(unzip_dir, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(unzip_dir, exist_ok=True)

# Unzip the files
for file in os.listdir(zip_dir):
    if file.endswith('.zip'):
        with zipfile.ZipFile(zip_dir + file, 'r') as zip_ref:
            zip_ref.extractall(unzip_dir)


# Define the directory where the .wav and TRANSCRIPT.csv files will be stored
final_dir = '../../data/raw/DAIC_WOZ_final/'

# Delete the directory if it already exists
shutil.rmtree(final_dir, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(final_dir, exist_ok=True)

# Move the .wav and TRANSCRIPT.csv files to the final directory
for root, dirs, files in os.walk(unzip_dir):
    for file in files:
        if file.endswith('.wav') or file.endswith('TRANSCRIPT.csv'):
            shutil.move(os.path.join(root, file), final_dir)
            

In [4]:
zip_dir = '../../data/raw/DAIC_WOZ/'
# Remove the zip files
shutil.rmtree(zip_dir, ignore_errors=True)

In [5]:
unzip_dir = '../../data/raw/DAIC_WOZ_unzipped/'
# Remove the unzipped directory
shutil.rmtree(unzip_dir, ignore_errors=True)

## Slice the audio data and match with transcript

Please download ffmpeg software and finish enviroment configuration.

In [6]:
import os
import pandas as pd

# Define the directory where the .wav and TRANSCRIPT.csv files will be stored
final_dir = '../../data/raw/DAIC_WOZ_final/'

# Define the directory where the sliced .wav and .txt files are stored
sliced_dir = '../../data/raw/DAIC_WOZ_sliced/'

# Delete the directory if it already exists
shutil.rmtree(sliced_dir, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(sliced_dir, exist_ok=True)

# List the files in the directory
os.listdir(final_dir)


# If need, delete this file '._487_TRANSCRIPT.csv'
try:
    os.remove('../../data/raw/DAIC_WOZ_final/._487_TRANSCRIPT.csv')
except FileNotFoundError:
    pass


# Load the TRANSCRIPT.csv file, which contains the transcriptions of the audio files
for file in os.listdir(final_dir):
    if file.endswith('TRANSCRIPT.csv'):
        # Define the audio file
        audio_file = file.replace('TRANSCRIPT.csv', 'AUDIO.wav')
        # Load the TRANSCRIPT.csv file
        transcript = pd.read_csv(final_dir + file, sep='\t', header='infer')
        # Set column names
        transcript.columns = ['start_time', 'stop_time', 'role', 'text']
        # Keep only the 'Participant' role
        transcript = transcript[transcript['role'] == 'Participant']
        # Keep only rows where the text is not empty
        transcript = transcript[transcript['text'].notnull()]
        # filter only start_time-stop_time >= 3
        transcript = transcript[transcript['stop_time'] - transcript['start_time'] >= 3]
        # Reset the index
        transcript.reset_index(drop=True, inplace=True)

        # For every row, save the text to a .txt file and slice the corresponding audio file
        for i in range(len(transcript)):            
            # Save the text to a .txt file
            with open(sliced_dir + f'{file.split("_")[0]}_{i}.txt', 'w') as f:
                f.write(transcript['text'][i])
            # Slice the audio file
            start_time = transcript['start_time'][i]
            stop_time = transcript['stop_time'][i]
            # Convert the start and stop times to the correct format: HH:MM:SS
            start_time = f'{int(start_time/3600):02d}:{int((start_time%3600)/60):02d}:{int(start_time%60):02d}'
            stop_time = f'{int(stop_time/3600):02d}:{int((stop_time%3600)/60):02d}:{int(stop_time%60):02d}'
            # Slice the audio file
            os.system(f'ffmpeg -i {final_dir + audio_file} -ss {start_time} -to {stop_time} {sliced_dir}{file.split("_")[0]}_{i}.wav')
        

## Audio and text data cleaning and augmentation

In [8]:
import re
def text_data_cleaning(file_path):
    # Read the text file, remove special characters and digits, convert the text to lowercase, and return the cleaned text to the text file
    with open(file_path, 'r') as f:
        text = f.read()
        # Remove special characters 
        text = re.sub(r'[^a-zA-Z\s\d]', '', text).strip()
        # No need to remove stopwords
        # Convert the text to lowercase
        text = text.lower()
    # Write the cleaned text to the text file
    with open(file_path, 'w') as f:
        f.write(text)
    return

In [8]:
for file in os.listdir(sliced_dir):
    if file.endswith('.txt'):
        text_data_cleaning(sliced_dir + file)

In [44]:
# !pip install -q noisereduce
# !pip install torchaudio
# !pip install -q librosa

In [9]:
import librosa
import numpy as np
import soundfile as sf
import noisereduce as nr
import random
import torch
import os

def clean_audio(input_path, output_path, target_sr=16000):
    """
    数据清洗：
    1. 采样率转换
    2. 响度归一化
    3. 降噪
    4. 静音移除
    """
    # 读取音频
    audio, sr = librosa.load(input_path, sr=None)
    
    # 采样率转换
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    
    # 响度归一化（标准化到 [-1, 1]）
    audio = audio / np.max(np.abs(audio))

    # 降噪处理
    audio = nr.reduce_noise(y=audio, sr=target_sr, prop_decrease=0.1)

    # 静音移除，如果全部为静音，则删除该音频
    # non_silent_intervals = librosa.effects.split(audio, top_db=20)  # top_db 控制静音阈值
    # try:
    #     audio = np.concatenate([audio[start:end] for start, end in non_silent_intervals])
    # except ValueError:
    #     os.remove(input_path)
    #     print(f'{input_path} is removed because it is silent.')
    #     return

    # 保存处理后的音频
    sf.write(output_path, audio, target_sr)

In [15]:
sliced_dir = '../../data/raw/DAIC_WOZ_sliced/'
cleaned_dir = '../../data/raw/DAIC_WOZ_cleaned/'
# Delete the directory if it already exists
try:
    shutil.rmtree(cleaned_dir)
except FileNotFoundError:
    pass

# Create the directory if it doesn't exist
os.makedirs(cleaned_dir, exist_ok=True)

for file in os.listdir(sliced_dir):
    if file.endswith('.wav'):
        clean_audio(sliced_dir + file, cleaned_dir + file)

C:\Users\Sora1874\AppData\Local\Temp\ipykernel_16368\3038589700.py:25: RuntimeWarning: invalid value encountered in divide
  audio = audio / np.max(np.abs(audio))


In [10]:
import numpy as np
import random

def add_noise(audio, snr_range=(25, 30)):
    """
    向音频添加高斯噪声，SNR 控制噪声大小
    - `snr_range`: (最小 SNR, 最大 SNR)
    """
    snr = random.uniform(*snr_range)  # 在设定范围内随机选择 SNR
    noise_std = np.std(audio) / (10 ** (snr / 20))  # 计算噪声标准差
    noise = np.random.normal(0, noise_std, audio.shape)  # 生成噪声
    audio_noisy = audio + noise  # 添加噪声
    return np.clip(audio_noisy, -1.0, 1.0)  # 避免超出 [-1, 1]

In [11]:
def augment_audio(input_path, output_path, target_sr=16000):
    """
    数据增强：
    1. 时间拉伸 (Time Stretching)
    2. 音高变化 (Pitch Shifting)
    3. 噪声注入 (Noise Augmentation)
    4. 频谱增强 (SpecAugment)
    """
    # 读取音频
    audio, sr = librosa.load(input_path, sr=target_sr)

    # 1. 时间拉伸
    if random.random() > 0.5:
        rate = random.uniform(0.95, 1.05)  # 在 0.95x ~ 1.05x 之间变化
        audio = librosa.effects.time_stretch(audio, rate=rate)
    
    # 2. 音高变化
    if random.random() > 0.5:
        steps = random.randint(-2, 2)  # 随机上下变 2 个半音
        audio = librosa.effects.pitch_shift(audio, sr=target_sr, n_steps=steps)

    # 3. 噪声注入
    if random.random() > 0.5:
        audio = add_noise(audio, snr_range=(25, 30))  # **控制噪声强度**

    # 保存增强后的音频
    sf.write(output_path, audio, target_sr)

# 示例调用
# clean_audio("input.wav", "cleaned.wav")
# augment_audio("cleaned.wav", "augmented.wav")

In [18]:
DAIC_WOZ_augmented_dir = '../../data/raw/DAIC_WOZ_augmented/'

# Delete the directory if it already exists
try:
    shutil.rmtree(DAIC_WOZ_augmented_dir)
except FileNotFoundError:
    pass

# Create the directory if it doesn't exist
os.makedirs(DAIC_WOZ_augmented_dir, exist_ok=True)

# Set the random seed
random.seed(42)

# Augment the audio files
for file in os.listdir('../../data/raw/DAIC_WOZ_cleaned/'):
    augment_audio(f'../../data/raw/DAIC_WOZ_cleaned/{file}', f'{DAIC_WOZ_augmented_dir}{file}')

In [19]:
# Move the augmented audio files and their corresponding text files to the processed directory
processed_audio_dir = '../../data/processed/DAIC_WOZ/audio/'
processed_text_dir = '../../data/processed/DAIC_WOZ/text/'
# Delete the directory if it already exists
shutil.rmtree(processed_audio_dir, ignore_errors=True)
shutil.rmtree(processed_text_dir, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(processed_audio_dir, exist_ok=True)
os.makedirs(processed_text_dir, exist_ok=True)

for file in os.listdir(DAIC_WOZ_augmented_dir):
    if file.endswith('.wav'):
        shutil.move(DAIC_WOZ_augmented_dir + file, processed_audio_dir + file)
        # Move the corresponding text file
        try:
            shutil.move(sliced_dir + file.replace('.wav', '.txt'), processed_text_dir + file.replace('.wav', '.txt'))
        except FileNotFoundError:
            pass

In [20]:
# Split the data into training, validation, and test sets at a ratio of 80:10:10, and copy the files to the corresponding splits directorie
# e.g. data/splits/DAIC_WOZ/{split}/audio
import random
import shutil
import os

processed_audio_dir = '../../data/processed/DAIC_WOZ/audio/'
processed_text_dir = '../../data/processed/DAIC_WOZ/text/'


data_split_list = ['train', 'val', 'test']
for split in data_split_list:
    # Delete the directory if it already exists
    shutil.rmtree(f'../../data/splits/DAIC_WOZ/{split}/audio/', ignore_errors=True)
    shutil.rmtree(f'../../data/splits/DAIC_WOZ/{split}/text/', ignore_errors=True)
    # Create the directory if it doesn't exist
    os.makedirs(f'../../data/splits/DAIC_WOZ/{split}/audio/', exist_ok=True)
    os.makedirs(f'../../data/splits/DAIC_WOZ/{split}/text/', exist_ok=True)


# Set the random seed
random.seed(42)

# Randomly assign the files to the training, validation, and test sets, and copy the files to the corresponding directories
for file in os.listdir(processed_audio_dir):
    split = random.choices(data_split_list, weights=[0.8, 0.1, 0.1], k=1)[0]
    shutil.copy(processed_audio_dir + file, f'../../data/splits/DAIC_WOZ/{split}/audio/{file}')
    try:
        shutil.copy(processed_text_dir + file.replace('.wav', '.txt'), f'../../data/splits/DAIC_WOZ/{split}/text/{file.replace(".wav", ".txt")}')
    except FileNotFoundError:
        pass

In [21]:
# Remove the directories that are no longer needed
try:
    shutil.rmtree('../../data/raw/DAIC_WOZ_unzipped/')
except FileNotFoundError:
    pass
try:
    shutil.rmtree('../../data/raw/DAIC_WOZ_final/')
except FileNotFoundError:
    pass
try:
    shutil.rmtree('../../data/raw/DAIC_WOZ_sliced/')
except FileNotFoundError:
    pass   
try:    
    shutil.rmtree('../../data/raw/DAIC_WOZ_cleaned/')
except FileNotFoundError:
    pass   
try:
    shutil.rmtree('../../data/raw/DAIC_WOZ_augmented/')
except FileNotFoundError:
    pass

# MELD preprocessing

In [2]:
import tarfile
import os
import shutil

# 解压缩 .tar.gz 文件
def extract_tar_gz(file_path, extract_path):
    with tarfile.open(file_path, 'r:gz') as tar:
        tar.extractall(path=extract_path)

# 示例
file_path = '../../data/raw/MELD.Raw.tar.gz'
extract_path = '../../data/raw/MELD_raw/'

# Delete the directory if it already exists
shutil.rmtree(extract_path, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

extract_tar_gz(file_path, extract_path)

In [3]:
# Countinue to extract the .tar.gz files
split_list = ['train', 'dev', 'test']
for split in split_list:
    file_path = f'../../data/raw/MELD_raw/MELD.Raw/{split}.tar.gz'
    # Delete the directory if it already exists
    shutil.rmtree(f'../../data/raw/MELD_raw/MELD.Raw/{split}/', ignore_errors=True)
    # Create the directory if it doesn't exist
    os.makedirs(f'../../data/raw/MELD_raw/MELD.Raw/{split}/', exist_ok=True)
    extract_path = f'../../data/raw/MELD_raw/MELD.Raw/{split}/'
    extract_tar_gz(file_path, extract_path)

In [4]:
# Move the dev_sent_emo.csv and test_sent_emo.csv to corresponding directories
try:
    shutil.move(f'../../data/raw/MELD_raw/MELD.Raw/dev_sent_emo.csv', f'../../data/raw/MELD_raw/MELD.Raw/dev/dev_sent_emo.csv')
except FileNotFoundError:
    pass
try:
    shutil.move(f'../../data/raw/MELD_raw/MELD.Raw/test_sent_emo.csv', f'../../data/raw/MELD_raw/MELD.Raw/test/test_sent_emo.csv')
except FileNotFoundError:
    pass

# Change the folder name to the correct name, e.g. dev_splits_complete -> dev_splits, output_repeated_splits_test -> test_splits
try:
    os.rename('../../data/raw/MELD_raw/MELD.Raw/dev/dev_splits_complete', '../../data/raw/MELD_raw/MELD.Raw/dev/dev_splits')
    os.rename('../../data/raw/MELD_raw/MELD.Raw/test/output_repeated_splits_test', '../../data/raw/MELD_raw/MELD.Raw/test/test_splits')
except FileNotFoundError:
    pass

# Delete files not in the correct format, e.g. not starting with 'dia'

try:
    for root, dirs, files in os.walk('../../data/raw/MELD_raw/MELD.Raw/test/test_splits/'):
        for file in files:
            if file.startswith('dia') == False:
                os.remove(os.path.join(root, file))
except FileNotFoundError:
    pass



In [5]:
# Label the splitted files with information from the .csv files, and move the labeled audio files to the labeled directory
import pandas as pd
import os
import shutil

split_list = ['train', 'dev', 'test']

# Convert the formats of the start and stop times format: from HH:MM:SS to seconds
def convert_time_MELD(time_1, time_2):
    h, m, s = time_1.split(':')
    ms = time_2
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

In [6]:
for split in split_list:
    # Define the directory where the raw files are stored
    MELD_raw_dir = f'../../data/raw/MELD_raw/MELD.Raw/{split}/{split}_splits/'

    # Define the directory where the labeled files are stored
    MELD_labeled_audio_dir = f'../../data/raw/MELD_labeled/{split}/audio/'
    MELD_labeled_text_dir = f'../../data/raw/MELD_labeled/{split}/text/'

    # Delete the directory if it already exists
    shutil.rmtree(MELD_labeled_audio_dir, ignore_errors=True)
    shutil.rmtree(MELD_labeled_text_dir, ignore_errors=True)

    # Create the directory if it doesn't exist
    os.makedirs(MELD_labeled_audio_dir, exist_ok=True)
    os.makedirs(MELD_labeled_text_dir, exist_ok=True)

    # Load the .csv file
    df = pd.read_csv(f'../../data/raw/MELD_raw/MELD.Raw/{split}/{split}_sent_emo.csv', header='infer')

    # For all the files in the MELD_raw_dir, modify the name to be original name+label, generate the transcripts txt file, and move the files to the MELD_labeled_dir with the correct emotion label
    for file in os.listdir(MELD_raw_dir):
        dia_num = file.split('_')[0].split('dia')[1]
        utt_num = file.split('_')[1].split('utt')[1].split('.')[0]
        # Filter if duration > 3s， if can't find the start_time, skip
        if len(df[(df['Dialogue_ID'] == int(dia_num)) & (df['Utterance_ID'] == int(utt_num))]) > 0:
            start_time = df[(df['Dialogue_ID'] == int(dia_num)) & (df['Utterance_ID'] == int(utt_num))]['StartTime'].values[0]
            end_time = df[(df['Dialogue_ID'] == int(dia_num)) & (df['Utterance_ID'] == int(utt_num))]['EndTime'].values[0]
            start_time_in_sec = convert_time_MELD( start_time.split(',')[0], start_time.split(',')[1])
            end_time_in_sec = convert_time_MELD( end_time.split(',')[0], end_time.split(',')[1])
            if end_time_in_sec - start_time_in_sec > 3:
                # Get the emotion label
                emotion = df[(df['Dialogue_ID'] == int(dia_num)) & (df['Utterance_ID'] == int(utt_num))]['Emotion'].values[0]
                # Modify the name
                audio_new_name = file.split('.')[0] + '_' + emotion + '.wav'
                text_new_name = file.split('.')[0] + '_' + emotion + '.txt'
                # retrieve the transcript
                transcript = df[(df['Dialogue_ID'] == int(dia_num)) & (df['Utterance_ID'] == int(utt_num))]['Utterance'].values[0]
                # Transfer the .mp4 file to .wav file and move the file
                os.system(f'ffmpeg -i {MELD_raw_dir + file} -ar 16000 {MELD_labeled_audio_dir + audio_new_name}')
                # Write the transcript to a text file
                with open(MELD_labeled_text_dir + text_new_name, 'w') as f:
                    f.write(transcript)




In [12]:
# Audio cleaning and augmentation
for split in split_list:
    # Define the directory where the cleaned audio files will be stored
    MELD_cleaned_dir = f'../../data/raw/MELD_cleaned/{split}/'

    # Delete the directory if it already exists
    shutil.rmtree(MELD_cleaned_dir, ignore_errors=True)
    # Create the directory if it doesn't exist
    os.makedirs(MELD_cleaned_dir, exist_ok=True)

    # Clean the audio files
    for file in os.listdir(f'../../data/raw/MELD_labeled/{split}/audio/'):
        clean_audio(f'../../data/raw/MELD_labeled/{split}/audio/{file}', f'{MELD_cleaned_dir}{file}')

    # Define the directory where the augmented audio files will be stored
    MELD_augmented_dir = f'../../data/raw/MELD_augmented/{split}/'

    # Delete the directory if it already exists
    shutil.rmtree(MELD_augmented_dir, ignore_errors=True)
    # Create the directory if it doesn't exist
    os.makedirs(MELD_augmented_dir, exist_ok=True)

    # Augment the audio files
    for file in os.listdir(MELD_cleaned_dir):
        augment_audio(MELD_cleaned_dir + file, MELD_augmented_dir + file)

In [13]:
# text data cleaning
for split in split_list:
    for file in os.listdir(f'../../data/raw/MELD_labeled/{split}/text/'):
        text_data_cleaning(f'../../data/raw/MELD_labeled/{split}/text/{file}')

In [14]:
# Move the audio and text files to the processed data directory
for split in split_list:
    processed_audio_dir = f'../../data/processed/MELD/{split}/audio/'
    processed_text_dir = f'../../data/processed/MELD/{split}/text/'
    # Delete the directory if it already exists
    shutil.rmtree(processed_audio_dir, ignore_errors=True)
    shutil.rmtree(processed_text_dir, ignore_errors=True)
    # Create the directory if it doesn't exist
    os.makedirs(processed_audio_dir, exist_ok=True)
    os.makedirs(processed_text_dir, exist_ok=True)
    for file in os.listdir(f'../../data/raw/MELD_augmented/{split}/'):
        if file.endswith('.wav'):
            shutil.move(f'../../data/raw/MELD_augmented/{split}/{file}', processed_audio_dir + file)
            # Move the corresponding text file
            try:
                shutil.move(f'../../data/raw/MELD_labeled/{split}/text/{file.replace(".wav", ".txt")}', processed_text_dir + file.replace('.wav', '.txt'))
            except FileNotFoundError:
                pass

In [29]:
# Rename the dev directory to val
try:
    os.rename('../../data/processed/MELD/dev', '../../data/processed/MELD/val')
except FileNotFoundError:
    pass

In [16]:
# Delete the directory if it already exists
shutil.rmtree('../../data/splits/MELD', ignore_errors=True)
# Copy the splitted MELD dataset to splits directory
shutil.copytree('../../data/processed/MELD', '../../data/splits/MELD')

'../../data/splits/MELD'

In [17]:
# Remove the directories that are no longer needed
shutil.rmtree('../../data/raw/MELD_raw/', ignore_errors=True)
shutil.rmtree('../../data/raw/MELD_labeled/', ignore_errors=True)
shutil.rmtree('../../data/raw/MELD_cleaned/', ignore_errors=True)
shutil.rmtree('../../data/raw/MELD_augmented/', ignore_errors=True)


## Organize MELD dataset

In [18]:
# Move the files of MELD dataset from the train and test directories to a collective directory
# e.g. data/splits/collective/{split}/audio
import shutil
import os

# Define the directory where the collective files will be stored
collective_audio_dir = '../../data/splits/collective/audio/'
collective_text_dir = '../../data/splits/collective/text/'

# Delete the directory if it already exists
shutil.rmtree(collective_audio_dir, ignore_errors=True)
shutil.rmtree(collective_text_dir, ignore_errors=True)
# Create the directory if it doesn't exist
os.makedirs(collective_audio_dir, exist_ok=True)
os.makedirs(collective_text_dir, exist_ok=True)

# Move the files to the collective directories
for split in ['train', 'val']:
    for file in os.listdir(f'../../data/splits/MELD/{split}/audio/'):
        shutil.move(f'../../data/splits/MELD/{split}/audio/{file}', collective_audio_dir + file)
        try:
            shutil.move(f'../../data/splits/MELD/{split}/text/{file.replace(".wav", ".txt")}', collective_text_dir + file.replace('.wav', '.txt'))
        except FileNotFoundError:
            pass


In [19]:
# Write a .csv file with the audio file names and their corresponding labels
import pandas as pd
import os

# Define the directory where the collective files are stored
collective_audio_dir = '../../data/splits/collective/audio/'

# Define the directory where the .csv file will be stored
csv_dir = '../../data/splits/collective/'

# Write the .csv file
data = []
for file in os.listdir(collective_audio_dir):
    data.append([file, file.split('_')[-1].split('.')[0]])
df = pd.DataFrame(data, columns=['file', 'label'])
df.to_csv(csv_dir + 'collective.csv', index=False)

In [20]:
# Rename the audio folder to audio_train
os.rename('../../data/splits/collective/audio', '../../data/splits/collective/train')

# Rename the csv file to trainLabels.csv
os.rename('../../data/splits/collective/collective.csv', '../../data/splits/collective/trainLabels.csv')

In [21]:
data_dir = '../../data/splits/collective/'

#@save
def read_csv_labels(fname):
    """Read `fname` to return a filename to label dictionary."""
    with open(fname, 'r') as f:
        # Skip the file header line (column name)
        lines = f.readlines()[1:]
    tokens = [l.rstrip().split(',') for l in lines]
    return dict(((name, label) for name, label in tokens))

labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
print('# training examples:', len(labels))
print('# classes:', len(set(labels.values())))

# training examples: 4260
# classes: 7


In [22]:
import collections
import math
#@save
def copyfile(filename, target_dir):
    """Copy a file into a target directory."""
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy(filename, target_dir)

#@save
def reorg_train_valid(data_dir, labels, valid_ratio):
    """Split the validation set out of the original training set."""
    # The number of examples of the class that has the fewest examples in the
    # training dataset
    n = collections.Counter(labels.values()).most_common()[-1][1]
    # The number of examples per class for the validation set
    n_valid_per_label = max(1, math.floor(n * valid_ratio))
    label_count = {}
    for train_file in os.listdir(os.path.join(data_dir, 'train')):
        label = labels[train_file] 
        fname = os.path.join(data_dir, 'train', train_file)
        copyfile(fname, os.path.join(data_dir, 'train_valid_test',
                                     'train_valid', label))
        if label not in label_count or label_count[label] < n_valid_per_label:
            copyfile(fname, os.path.join(data_dir, 'train_valid_test',
                                         'valid', label))
            label_count[label] = label_count.get(label, 0) + 1
        else:
            copyfile(fname, os.path.join(data_dir, 'train_valid_test',
                                         'train', label))
    return n_valid_per_label

In [23]:
#@save
def reorg_test(data_dir):
    """Organize the testing set for data loading during prediction."""
    for test_file in os.listdir(os.path.join(data_dir, 'test')):
        copyfile(os.path.join(data_dir, 'test', test_file),
                 os.path.join(data_dir, 'train_valid_test', 'test',
                              'unknown'))

In [24]:
labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
valid_ratio = 0.1
reorg_train_valid(data_dir, labels, valid_ratio)

12

In [25]:
# Move the files in the train and valid directories to the splits directory

# Define the directory where the files are stored
for split in ['valid']:
    for label in os.listdir(f'../../data/splits/collective/train_valid_test/{split}/'):
        shutil.move(f'../../data/splits/collective/train_valid_test/{split}/{label}', f'../../data/splits/MELD/{split}/{label}')
        # Extract the files in the subdirectories to the parent audio directory
        # Create the directory if it doesn't exist
        os.makedirs(f'../../data/splits/MELD/{split}/audio/', exist_ok=True)
        os.makedirs(f'../../data/splits/MELD/{split}/text/', exist_ok=True)
        for file in os.listdir(f'../../data/splits/MELD/{split}/{label}'):
            shutil.move(f'../../data/splits/MELD/{split}/{label}/{file}', f'../../data/splits/MELD/{split}/audio/{file}')
            # Try to move the corresponding text files
            try:
                shutil.move(f'../../data/splits/collective/text/{file.replace(".wav", ".txt")}', f'../../data/splits/MELD/{split}/text/{file.replace(".wav", ".txt")}')
            except FileNotFoundError:
                pass
        # Remove the subdirectories
        shutil.rmtree(f'../../data/splits/MELD/{split}/{label}')



In [26]:
# Try to move the corresponding text files
os.makedirs(f'../../data/splits/MELD/valid/text/', exist_ok=True)
for file in os.listdir(f'../../data/splits/MELD/valid/audio/'):
    try:
        shutil.move(f'../../data/splits/collective/text/{file.replace(".wav", ".txt")}', f'../../data/splits/MELD/valid/text/{file.replace(".wav", ".txt")}')
    except FileNotFoundError:
        pass

In [27]:
# Rename the valid directory to val
# Delete the directory if it already exists
shutil.rmtree('../../data/splits/MELD/val/', ignore_errors=True)
# Create the directory if it doesn't exist
os.rename('../../data/splits/MELD/valid', '../../data/splits/MELD/val')

In [28]:
# Delete the collective directory
shutil.rmtree('../../data/splits/collective/', ignore_errors=True)